# Simple Treatment Effect Notebook: HbA1c and Anemia Outcomes

This beginner-friendly notebook simulates a simple healthcare study comparing **Treatment A** and **Treatment B**.

We will analyze two outcomes:

1. **HbA1c change**
   - HbA1c is a common diabetes measure
   - more negative change means a bigger reduction, which is usually better

2. **Hemoglobin change**
   - hemoglobin is related to anemia
   - more positive change means improvement in anemia status

We will:

- create simple simulated data
- inspect the dataset
- compare treatment groups
- run **t-tests**
- run **Mann–Whitney U tests**
- calculate **95% confidence intervals**
- interpret whether the treatment effect looks meaningful

## Clinical scenario

Imagine we are evaluating two treatments in patients with diabetes who may also have anemia.

### Outcomes

**Outcome 1: `hba1c_change`**
- negative values mean HbA1c went down
- a larger drop is better

**Outcome 2: `hemoglobin_change`**
- positive values mean hemoglobin increased
- that may suggest improvement in anemia

This is a simulated dataset for learning only.

## Step 0: Setup

We import the main Python libraries we need.

- `numpy` for numbers and random simulation
- `pandas` for tables
- `matplotlib` and `seaborn` for plots
- `scipy.stats` for t-tests and Mann–Whitney tests
- `statsmodels` for confidence intervals

In [ ]:
# STEP 0: IMPORT LIBRARIES

try:
    import google.colab
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    !pip -q install numpy pandas scipy statsmodels seaborn matplotlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.stats.api as sms

np.random.seed(42)
sns.set(style="whitegrid")
pd.set_option("display.max_columns", 50)

print("Setup complete. Running in Colab:", IN_COLAB)

## Step 1: Create a simple simulated dataset

We will simulate 2 groups:

- **Treatment A**
- **Treatment B**

### Design idea

For this teaching example:

- Treatment B will produce a slightly larger average drop in HbA1c
- Treatment B will also produce a slightly larger average rise in hemoglobin

This lets us practice interpreting whether the effect is:

- statistically significant
- clinically meaningful

In [ ]:
# STEP 1: SIMULATE A SIMPLE TREATMENT STUDY

def simulate_diabetes_anemia_study(n_per_group=80):
    """
    Create a simple simulated study with two treatments and two outcomes.

    hba1c_change:
        negative is better because HbA1c decreases

    hemoglobin_change:
        positive is better because hemoglobin improves
    """

    # Treatment A: smaller average HbA1c reduction
    hba1c_a = np.random.normal(loc=-0.6, scale=0.7, size=n_per_group)

    # Treatment B: larger average HbA1c reduction
    hba1c_b = np.random.normal(loc=-1.0, scale=0.7, size=n_per_group)

    # Treatment A: smaller average hemoglobin improvement
    hgb_a = np.random.normal(loc=0.2, scale=0.8, size=n_per_group)

    # Treatment B: larger average hemoglobin improvement
    hgb_b = np.random.normal(loc=0.6, scale=0.8, size=n_per_group)

    df = pd.DataFrame({
        "treatment": ["A"] * n_per_group + ["B"] * n_per_group,
        "hba1c_change": np.concatenate([hba1c_a, hba1c_b]),
        "hemoglobin_change": np.concatenate([hgb_a, hgb_b])
    })

    return df

df = simulate_diabetes_anemia_study(n_per_group=90)
df.head()

## Step 2: Inspect the data

Before doing statistics, always inspect the data first.

We want to check:

- how many rows and columns we have
- whether the variables look correct
- the summary statistics

In [ ]:
# STEP 2: BASIC INSPECTION

print("Rows, Columns:", df.shape)

print("\nData types:")
print(df.dtypes)

print("\nSummary statistics:")
display(df.describe().T)

## Step 3: Grouped summary

A grouped summary helps us quickly compare Treatment A and Treatment B.

We will calculate:

- number of patients
- mean HbA1c change
- standard deviation of HbA1c change
- mean hemoglobin change
- standard deviation of hemoglobin change

In [ ]:
# STEP 3: GROUPED SUMMARY

group_summary = df.groupby("treatment").agg(
    n_patients=("treatment", "count"),
    mean_hba1c_change=("hba1c_change", "mean"),
    sd_hba1c_change=("hba1c_change", "std"),
    mean_hemoglobin_change=("hemoglobin_change", "mean"),
    sd_hemoglobin_change=("hemoglobin_change", "std")
)

display(group_summary)

## Step 4: Visualize the distributions

We will look at simple plots for both outcomes.

This helps us understand:

- center
- spread
- overlap between groups
- possible outliers

In [ ]:
# STEP 4A: HISTOGRAM FOR HbA1c CHANGE

plt.figure(figsize=(8, 4))
sns.histplot(data=df, x="hba1c_change", hue="treatment", kde=True, bins=25)
plt.title("Distribution of HbA1c Change by Treatment")
plt.show()

In [ ]:
# STEP 4B: HISTOGRAM FOR HEMOGLOBIN CHANGE

plt.figure(figsize=(8, 4))
sns.histplot(data=df, x="hemoglobin_change", hue="treatment", kde=True, bins=25)
plt.title("Distribution of Hemoglobin Change by Treatment")
plt.show()

In [ ]:
# STEP 4C: BOXPLOTS FOR BOTH OUTCOMES

plt.figure(figsize=(6, 4))
sns.boxplot(data=df, x="treatment", y="hba1c_change")
plt.title("Boxplot of HbA1c Change by Treatment")
plt.show()

plt.figure(figsize=(6, 4))
sns.boxplot(data=df, x="treatment", y="hemoglobin_change")
plt.title("Boxplot of Hemoglobin Change by Treatment")
plt.show()

## Step 5: Prepare the groups

Most statistical tests compare two separate arrays:

- one for Treatment A
- one for Treatment B

In [ ]:
# STEP 5: EXTRACT GROUPS FOR EACH OUTCOME

hba1c_a = df[df["treatment"] == "A"]["hba1c_change"].dropna()
hba1c_b = df[df["treatment"] == "B"]["hba1c_change"].dropna()

hgb_a = df[df["treatment"] == "A"]["hemoglobin_change"].dropna()
hgb_b = df[df["treatment"] == "B"]["hemoglobin_change"].dropna()

print("HbA1c group sizes:", len(hba1c_a), len(hba1c_b))
print("Hemoglobin group sizes:", len(hgb_a), len(hgb_b))

## Step 6: Compare HbA1c change between groups

### Question
Did Treatment B reduce HbA1c more than Treatment A?

### Statistical idea

We will run:

- **Welch's t-test** for mean difference
- **Mann–Whitney U test** as a nonparametric alternative
- **95% CI** for the mean difference

Important:

For HbA1c change, **more negative is better**.
So if `B - A` is negative, that means Treatment B reduced HbA1c more.

In [ ]:
# STEP 6A: WELCH'S T-TEST FOR HbA1c CHANGE

t_hba1c, p_hba1c = stats.ttest_ind(hba1c_a, hba1c_b, equal_var=False)

print("Welch's t-test for HbA1c change")
print("t-statistic:", round(t_hba1c, 3))
print("p-value:", round(p_hba1c, 4))

In [ ]:
# STEP 6B: MANN-WHITNEY U TEST FOR HbA1c CHANGE

u_hba1c, u_p_hba1c = stats.mannwhitneyu(hba1c_a, hba1c_b, alternative="two-sided")

print("Mann-Whitney U test for HbA1c change")
print("U statistic:", round(u_hba1c, 3))
print("p-value:", round(u_p_hba1c, 4))

In [ ]:
# STEP 6C: 95% CONFIDENCE INTERVAL FOR MEAN DIFFERENCE IN HbA1c CHANGE

# We calculate (B - A)
cm_hba1c = sms.CompareMeans(sms.DescrStatsW(hba1c_b), sms.DescrStatsW(hba1c_a))
ci_low_hba1c, ci_high_hba1c = cm_hba1c.tconfint_diff(usevar="unequal")

mean_a_hba1c = hba1c_a.mean()
mean_b_hba1c = hba1c_b.mean()
mean_diff_hba1c = mean_b_hba1c - mean_a_hba1c

print("Mean HbA1c change - Treatment A:", round(mean_a_hba1c, 3))
print("Mean HbA1c change - Treatment B:", round(mean_b_hba1c, 3))
print("Mean difference (B - A):", round(mean_diff_hba1c, 3))
print("95% CI:", (round(ci_low_hba1c, 3), round(ci_high_hba1c, 3)))

## How to interpret HbA1c effect meaningfully

For HbA1c:

- a **negative** mean difference `(B - A)` means Treatment B lowered HbA1c more
- the farther below 0, the stronger the additional reduction
- if the 95% CI does not cross 0, that supports a real difference

A simple clinical rule of thumb in many teaching examples is:

- around **0.3 to 0.5 HbA1c units** may already be practically noticeable
- larger reductions may be more clinically meaningful depending on context

This is a simplified teaching rule, not a treatment guideline.

In [ ]:
# STEP 6D: SIMPLE INTERPRETATION FOR HbA1c

print("Interpretation for HbA1c:")

if p_hba1c < 0.05:
    print("- There is a statistically significant difference between treatments.")
else:
    print("- There is no statistically significant difference between treatments.")

if mean_diff_hba1c < -0.5:
    print("- Treatment B appears meaningfully better because it lowers HbA1c by more than 0.5 additional units on average.")
elif mean_diff_hba1c < -0.3:
    print("- Treatment B appears modestly but possibly meaningfully better for HbA1c reduction.")
else:
    print("- The additional HbA1c reduction may be small in practical terms.")

if ci_low_hba1c < 0 and ci_high_hba1c < 0:
    print("- The confidence interval stays below 0, which supports better HbA1c reduction with Treatment B.")
else:
    print("- The confidence interval includes 0, so uncertainty remains about the true HbA1c difference.")

## Step 7: Compare hemoglobin change between groups

### Question
Did Treatment B improve hemoglobin more than Treatment A?

For hemoglobin change:

- **positive** values are better
- if `B - A` is positive, Treatment B improved hemoglobin more

In [ ]:
# STEP 7A: WELCH'S T-TEST FOR HEMOGLOBIN CHANGE

t_hgb, p_hgb = stats.ttest_ind(hgb_a, hgb_b, equal_var=False)

print("Welch's t-test for hemoglobin change")
print("t-statistic:", round(t_hgb, 3))
print("p-value:", round(p_hgb, 4))

In [ ]:
# STEP 7B: MANN-WHITNEY U TEST FOR HEMOGLOBIN CHANGE

u_hgb, u_p_hgb = stats.mannwhitneyu(hgb_a, hgb_b, alternative="two-sided")

print("Mann-Whitney U test for hemoglobin change")
print("U statistic:", round(u_hgb, 3))
print("p-value:", round(u_p_hgb, 4))

In [ ]:
# STEP 7C: 95% CONFIDENCE INTERVAL FOR MEAN DIFFERENCE IN HEMOGLOBIN CHANGE

# We calculate (B - A)
cm_hgb = sms.CompareMeans(sms.DescrStatsW(hgb_b), sms.DescrStatsW(hgb_a))
ci_low_hgb, ci_high_hgb = cm_hgb.tconfint_diff(usevar="unequal")

mean_a_hgb = hgb_a.mean()
mean_b_hgb = hgb_b.mean()
mean_diff_hgb = mean_b_hgb - mean_a_hgb

print("Mean hemoglobin change - Treatment A:", round(mean_a_hgb, 3))
print("Mean hemoglobin change - Treatment B:", round(mean_b_hgb, 3))
print("Mean difference (B - A):", round(mean_diff_hgb, 3))
print("95% CI:", (round(ci_low_hgb, 3), round(ci_high_hgb, 3)))

## How to interpret hemoglobin effect meaningfully

For hemoglobin:

- a **positive** difference means Treatment B improved hemoglobin more
- if the 95% CI stays above 0, that supports a real benefit

In a simple teaching context:

- around **0.3 to 0.5 g/dL** may be modest
- larger increases may look more meaningful

This is only a beginner teaching interpretation, not a clinical recommendation.

In [ ]:
# STEP 7D: SIMPLE INTERPRETATION FOR HEMOGLOBIN

print("Interpretation for hemoglobin:")

if p_hgb < 0.05:
    print("- There is a statistically significant difference between treatments.")
else:
    print("- There is no statistically significant difference between treatments.")

if mean_diff_hgb > 0.5:
    print("- Treatment B appears meaningfully better because it increases hemoglobin by more than 0.5 additional units on average.")
elif mean_diff_hgb > 0.3:
    print("- Treatment B appears modestly but possibly meaningfully better for hemoglobin improvement.")
else:
    print("- The additional hemoglobin improvement may be small in practical terms.")

if ci_low_hgb > 0 and ci_high_hgb > 0:
    print("- The confidence interval stays above 0, which supports better hemoglobin improvement with Treatment B.")
else:
    print("- The confidence interval includes 0, so uncertainty remains about the true hemoglobin difference.")

## Step 8: Final plain-language summary

Now we create a short report-like interpretation.

This is the kind of summary you might write in a student assignment or beginner report.

In [ ]:
# STEP 8: PLAIN-LANGUAGE SUMMARY

summary_hba1c = (
    f"HbA1c change: Treatment A mean = {mean_a_hba1c:.2f}, "
    f"Treatment B mean = {mean_b_hba1c:.2f}, "
    f"mean difference (B - A) = {mean_diff_hba1c:.2f} "
    f"(95% CI {ci_low_hba1c:.2f} to {ci_high_hba1c:.2f}, "
    f"Welch p = {p_hba1c:.4f})."
)

summary_hgb = (
    f"Hemoglobin change: Treatment A mean = {mean_a_hgb:.2f}, "
    f"Treatment B mean = {mean_b_hgb:.2f}, "
    f"mean difference (B - A) = {mean_diff_hgb:.2f} "
    f"(95% CI {ci_low_hgb:.2f} to {ci_high_hgb:.2f}, "
    f"Welch p = {p_hgb:.4f})."
)

print(summary_hba1c)
print()
print(summary_hgb)

## Step 9: Practice ideas

Try these by yourself:

1. change the sample size from 90 to 40 or 150
2. make the treatment effect smaller
3. rerun the notebook
4. see how the p-values and confidence intervals change

This helps you learn a very important lesson:

**sample size and effect size both matter**

In [ ]:
# OPTIONAL PRACTICE CELL

# Example:
# df = simulate_diabetes_anemia_study(n_per_group=150)

print("Try your own experiments here.")

## Final take-home ideas

Do not memorize the code line by line.

Instead, remember the pattern:

### For each outcome
1. understand what direction is better
2. inspect the data
3. compare group means
4. run a t-test
5. run Mann–Whitney if you want a nonparametric comparison
6. compute a confidence interval
7. decide whether the effect is statistically and practically meaningful

That pattern is much more important than memorizing function names.